# Fake-or-Real(FoR) — Colab 딥페이크 음성 탐지 워크플로우

Kaggle의 [The Fake-or-Real (FoR) Dataset](https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset)을 Colab으로 내려받아 다음 순서로 실험합니다.

`EDA → Tiny baseline → AASIST → RawBoost → XLS-R → XLS-R+AASIST-inspired → TCM → speech/spectral multi-branch → ensemble → threshold/calibration`

내부 라벨은 끝까지 `0=fake`, `1=real`로 통일합니다. 체크포인트는 단일 validation EER이 아니라 여러 validation domain의 평균·최악 EER을 함께 사용해 선택합니다.

## 확인된 데이터 특성

- 195,000개 이상의 실제/합성 음성 발화
- `for-original`: 원본 수집 파일
- `for-norm`: 성별·클래스 균형 및 sample rate/volume/channel 정규화
- `for-2sec`: `for-norm`을 2초로 절단
- `for-rerec`: 통화/음성 메시지 상황을 모사한 재녹음 버전
- Kaggle 표시 용량 약 20.22 GB, WAV와 MP3 혼합, LGPL-3.0

중요: FoR는 기본적으로 **speech real/fake 데이터셋**입니다. 음악 단독/음성+음악, VC, audio inpainting의 공식 정답 레이블은 없습니다. 아래 EDA의 `content_hint`와 `attack_hint`는 탐색용 휴리스틱이며 학습 정답으로 사용하지 않습니다.


## 실행 원칙

1. 별도의 quick/sample 학습 모드는 사용하지 않습니다. 네 variant의 **전체 training 파일**을 매 epoch 한 번씩 모두 읽습니다.
2. `training`만 최적화에 사용하고 `validation`으로 checkpoint·가중치·threshold를 선택합니다.
3. 최적 checkpoint를 다시 불러와 네 variant의 **전체 testing 파일**을 평가합니다.
4. 학습이 긴 SSL 모델은 `last.pt`에서 epoch 단위로 자동 재개합니다.
5. FoR의 파일 번호는 split마다 다시 사용될 수 있으므로, 공식 split을 `group_id`의 네임스페이스로 삼고 같은 split 안의 variant 변환본을 묶습니다.

“최적 파라미터”는 GPU·split·목표 지표에 따라 달라집니다. 이 파일은 T4/L4에서 시작하기 좋은 모델별 설정과 validation 기반 선택 코드를 제공합니다. TEST 점수를 보고 다시 설정하면 정상적인 최종 평가가 아닙니다.


## 0. Colab 설치

GPU 런타임을 선택하세요. 설치 셀 후 import 오류가 남으면 런타임을 한 번 재시작합니다.


In [ ]:
import subprocess
import sys

packages = [
    "kaggle>=1.7", "transformers>=4.48,<5", "accelerate>=1.2",
    "librosa>=0.10.2", "soundfile>=0.12", "scikit-learn>=1.4",
    "seaborn>=0.13", "pandas>=2.0", "scipy>=1.11",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("필수 패키지 설치 완료")


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path
from types import SimpleNamespace

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
    log_loss, roc_auc_score, roc_curve,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

from google.colab import drive
drive.mount("/content/drive")

CFG = SimpleNamespace(
    seed=42,
    sample_rate=16_000,
    eda_per_group=15,
    num_workers=4,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/deepvoice_for")
DATA_ROOT = Path("/content/for_dataset")       # 큰 오디오는 Colab 로컬 SSD 사용
REPO_ROOT = Path("/content/for_repos")
RUN_ROOT = DRIVE_ROOT / "runs"
for directory in (DRIVE_ROOT, DATA_ROOT, REPO_ROOT, RUN_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
print("free disk GB:", round(shutil.disk_usage("/content").free / 1024**3, 1))


## 1. Kaggle에서 FoR 다운로드

전체 데이터가 크므로 압축 해제 위치는 Drive가 아니라 `/content/for_dataset`입니다. 학습 결과와 manifest만 Drive에 저장합니다.

Colab Secrets에 `KAGGLE_USERNAME`, `KAGGLE_KEY`를 등록하는 방법을 권장합니다. 없으면 셀이 `kaggle.json` 업로드를 요청합니다. 인증값을 출력하거나 Drive에 저장하지 않습니다.


In [ ]:
KAGGLE_DATASET = "mohammedabdeldayem/the-fake-or-real-dataset"
DOWNLOAD_DATA = True
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".amr"}

existing_audio = next((p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES), None)

if DOWNLOAD_DATA and existing_audio is None:
    from google.colab import files, userdata

    kaggle_api_token = kaggle_user = kaggle_key = None
    try:
        # Colab Secrets의 사용자 지정 이름을 Kaggle CLI 공식 환경변수에 연결합니다.
        kaggle_api_token = userdata.get("KAGGLE_API_KEY")
    except Exception:
        pass
    if not kaggle_api_token:
        try:
            kaggle_user = userdata.get("KAGGLE_USERNAME")
            kaggle_key = userdata.get("KAGGLE_KEY")
        except Exception:
            pass

    if kaggle_api_token:
        os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token
        print("Colab Secret KAGGLE_API_KEY를 사용합니다.")
    elif kaggle_user and kaggle_key:
        os.environ["KAGGLE_USERNAME"] = kaggle_user
        os.environ["KAGGLE_KEY"] = kaggle_key
    else:
        print("Kaggle Settings에서 받은 kaggle.json을 업로드하세요.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json이 업로드되지 않았습니다.")
        credential_dir = Path("/root/.kaggle")
        credential_dir.mkdir(parents=True, exist_ok=True)
        credential_path = credential_dir / "kaggle.json"
        credential_path.write_bytes(uploaded["kaggle.json"])
        credential_path.chmod(0o600)

    print("Kaggle file inventory:")
    subprocess.run(
        ["kaggle", "datasets", "files", KAGGLE_DATASET, "--page-size", "200"],
        check=True,
    )
    print("약 20 GB 데이터 다운로드/압축 해제를 시작합니다.")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "-p", str(DATA_ROOT), "--unzip", "--quiet"],
        check=True,
    )

audio_count = sum(1 for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES)
print("audio files:", f"{audio_count:,}")
if audio_count == 0:
    raise RuntimeError("오디오를 찾지 못했습니다. 다운로드 로그와 DATA_ROOT를 확인하세요.")


## 2. 폴더를 자동 해석해 manifest 생성

Kaggle 버전 또는 압축 내부 디렉터리명이 조금 달라도 `variant / split / real·fake` 토큰을 찾아 인덱싱합니다. 공식 split이 없을 때 임의로 넘어가지 않고 오류를 발생시킵니다.

FoR의 `file1`, `file100` 같은 이름은 training/validation/testing마다 다시 등장할 수 있어 전역 고유 ID가 아닙니다. 따라서 이름 충돌만으로 데이터 누수라고 판정하지 않습니다. 공식 split을 신뢰하고, `group_id = split + label + 정규화 파일번호`로 만들어 같은 split 안의 `for-original / for-norm / for-2sec / for-rerec` 대응본만 그룹화합니다.


In [ ]:
variant_patterns = [
    ("for-rerec", ("for-rerec", "for-rerecorded")),
    ("for-2sec", ("for-2sec", "for-2seconds", "for-2second")),
    ("for-original", ("for-original",)),
    ("for-norm", ("for-norm", "for-normalized")),
]
split_tokens = {
    "train": {"train", "training"},
    "validation": {"val", "valid", "validation", "dev"},
    "test": {"test", "testing", "eval", "evaluation"},
}

rows = []
for path in tqdm(DATA_ROOT.rglob("*"), desc="FoR file index"):
    if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
        continue
    lowered = path.as_posix().lower()
    parts = {part.lower() for part in path.parts}
    variant = next((name for name, patterns in variant_patterns if any(pattern in lowered for pattern in patterns)), "unknown")
    split = next((name for name, tokens in split_tokens.items() if parts & tokens), "unknown")
    has_real = bool(parts & {"real", "bonafide", "genuine"})
    has_fake = bool(parts & {"fake", "spoof", "synthetic"})
    label = 1 if has_real and not has_fake else 0 if has_fake and not has_real else -1

    attack_hint = "real" if label == 1 else "synthetic_unknown"
    for token, attack in (
        ("wavenet", "tts_wavenet"), ("deepvoice", "tts_deepvoice"),
        ("tts", "tts_other"), ("voice_conversion", "voice_conversion"),
        ("_vc_", "voice_conversion"), ("inpaint", "audio_inpainting"),
    ):
        if token in lowered:
            attack_hint = attack
            break

    # file1004.wav_16k.wav_norm...처럼 처리 suffix가 달라도 같은 번호를 묶습니다.
    # 파일 번호는 각 공식 split에서 재사용되므로 split을 반드시 namespace에 포함합니다.
    stem = path.stem.lower()
    file_number = re.match(r"^(file\d+)", stem)
    canonical_stem = file_number.group(1) if file_number else stem
    source_key = f"{label}:{canonical_stem}"
    group_id = f"{split}:{source_key}"
    rows.append({
        "id": f"{variant}:{split}:{path.stem}", "path": str(path),
        "label": label, "class_name": "real" if label == 1 else "fake",
        "variant": variant, "split": split, "source_key": source_key,
        "group_id": group_id,
        "attack_hint": attack_hint, "content_ground_truth": "speech",
        "extension": path.suffix.lower(),
    })

manifest = pd.DataFrame(rows)
bad = manifest[(manifest["variant"] == "unknown") | (manifest["split"] == "unknown") | (manifest["label"] < 0)]
if len(bad):
    display(bad[["path", "variant", "split", "label"]].head(20))
    raise ValueError(f"폴더 구조를 해석하지 못한 파일 {len(bad):,}개. 위 토큰 표를 실제 구조에 맞게 수정하세요.")

if manifest.duplicated("id").any():
    manifest["id"] = manifest["id"] + ":" + manifest.groupby("id").cumcount().astype(str)

# 진단용: 같은 이름이 여러 split에 있는 것은 FoR의 정상적인 번호 재사용입니다.
# source_key는 전역 발화 ID가 아니므로 여기서는 hard error를 발생시키지 않습니다.
cross_split_names = manifest.groupby("source_key")["split"].nunique()
cross_split_names = cross_split_names[cross_split_names > 1]
if len(cross_split_names):
    print(
        f"정보: {len(cross_split_names):,}개 파일 번호가 여러 공식 split에서 재사용됩니다. "
        "이는 누수 판정 기준이 아니며 split-qualified group_id를 사용합니다."
    )
    collision_examples = manifest[
        manifest["source_key"].isin(cross_split_names.index[:10])
    ][["source_key", "variant", "split", "class_name", "path"]]
    display(collision_examples.sort_values(["source_key", "split", "variant"]).head(40))

if manifest.groupby("group_id")["split"].nunique().max() != 1:
    raise RuntimeError("내부 오류: split-qualified group_id 생성에 실패했습니다.")

# 다운로드가 불완전하거나 0 byte인 파일은 학습 전에 명시적으로 제외합니다.
file_sizes = []
for audio_path in tqdm(manifest["path"], desc="file availability check"):
    try:
        candidate = Path(audio_path)
        file_sizes.append(candidate.stat().st_size if candidate.is_file() else -1)
    except OSError:
        file_sizes.append(-1)
manifest["file_bytes"] = file_sizes
unavailable_files = manifest[manifest["file_bytes"] <= 0].copy()
if len(unavailable_files):
    unavailable_path = DRIVE_ROOT / "unavailable_audio_files.csv"
    unavailable_files.to_csv(unavailable_path, index=False)
    print(f"경고: 없거나 0 byte인 파일 {len(unavailable_files):,}개를 제외합니다.")
    print("saved:", unavailable_path)
    display(unavailable_files[["path", "variant", "split", "class_name", "file_bytes"]].head(20))
    manifest = manifest[manifest["file_bytes"] > 0].reset_index(drop=True)

manifest_path = DRIVE_ROOT / "for_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(pd.crosstab([manifest["variant"], manifest["split"]], manifest["class_name"], margins=True))
print("manifest:", manifest_path, manifest.shape)


## 3. EDA 표본 설계

전체 17만여 파일을 고급 STFT 분석하면 시간이 오래 걸립니다. `variant × split × label`마다 동일 개수만 추출해 domain 크기 차이가 EDA를 지배하지 않게 합니다. `CFG.eda_per_group`을 늘리면 더 안정적인 통계를 얻습니다.


In [ ]:
sampled = []
for _, group in manifest.groupby(["variant", "split", "label"], sort=False):
    sampled.append(group.sample(min(CFG.eda_per_group, len(group)), random_state=CFG.seed))
eda_sample = pd.concat(sampled, ignore_index=True)
print("EDA sample:", len(eda_sample))
display(pd.crosstab([eda_sample["variant"], eda_sample["split"]], eda_sample["class_name"]))


## 4. 기본·고급 음향 EDA

다음 값을 한 번에 계산합니다.

- duration, sample rate, channel, format/subtype, 추정 bitrate
- RMS, clipping, zero-crossing, silence 비율, SNR proxy
- 6 kHz 이상 고주파 에너지 비율
- group delay와 instantaneous frequency 변동량
- harmonic ratio, onset rate, spectral flatness 기반 `speech/music/mixed` 참고값

Phase 계열 통계는 codec과 resampling에도 크게 변하므로 모델 정답 특징이 아니라 domain 차이와 leakage를 찾는 용도로만 사용합니다.


In [ ]:
def audio_artifacts(path: str, max_seconds: float = 12.0) -> dict:
    try:
        info = sf.info(path)
        audio, sr = sf.read(path, dtype="float32", always_2d=True)
        channels = info.channels
        audio_format, subtype = info.format, info.subtype
        duration = info.duration
        mono = audio.mean(axis=1)
    except Exception:
        audio, sr = librosa.load(path, sr=None, mono=False)
        channels = 1 if audio.ndim == 1 else audio.shape[0]
        mono = audio if audio.ndim == 1 else audio.mean(axis=0)
        duration = librosa.get_duration(y=mono, sr=sr)
        audio_format, subtype = Path(path).suffix.lower().lstrip("."), "unknown"

    mono = np.asarray(mono[: int(sr * max_seconds)], dtype=np.float32)
    if not len(mono):
        raise ValueError("empty audio")
    analysis = librosa.resample(mono, orig_sr=sr, target_sr=CFG.sample_rate) if sr != CFG.sample_rate else mono
    rms_frames = librosa.feature.rms(y=analysis, frame_length=1024, hop_length=256)[0]
    reference = max(float(rms_frames.max()), 1e-8)
    silence_ratio = float(np.mean(20 * np.log10(np.maximum(rms_frames, 1e-8) / reference) < -40))
    snr_proxy = 20 * np.log10((np.percentile(rms_frames, 90) + 1e-8) / (np.percentile(rms_frames, 10) + 1e-8))

    stft = librosa.stft(analysis, n_fft=1024, hop_length=256, win_length=1024)
    magnitude = np.abs(stft)
    frequencies = librosa.fft_frequencies(sr=CFG.sample_rate, n_fft=1024)
    high_ratio = float(magnitude[frequencies >= 6000].sum() / max(magnitude.sum(), 1e-8))
    phase = np.unwrap(np.angle(stft), axis=0)
    group_delay = -np.diff(phase, axis=0)
    instantaneous_frequency = np.diff(np.unwrap(np.angle(stft), axis=1), axis=1)
    flatness = float(librosa.feature.spectral_flatness(S=magnitude).mean())
    harmonic = librosa.effects.harmonic(analysis)
    harmonic_ratio = float(np.sum(harmonic**2) / max(np.sum(analysis**2), 1e-8))
    onset_rate = float(len(librosa.onset.onset_detect(y=analysis, sr=CFG.sample_rate)) / max(len(analysis) / CFG.sample_rate, 1e-6))

    if harmonic_ratio > 0.62 and onset_rate > 1.8 and flatness < 0.10:
        content_hint = "music_like"
    elif harmonic_ratio > 0.48 and onset_rate > 1.1:
        content_hint = "mixed_like"
    else:
        content_hint = "speech_like"

    codec = "g711" if str(subtype).upper() in {"ULAW", "ALAW"} else "amr" if Path(path).suffix.lower() == ".amr" else str(audio_format).lower()
    return {
        "duration": float(duration), "sample_rate": int(sr), "channels": int(channels),
        "format": str(audio_format), "subtype": str(subtype), "codec_hint": codec,
        "bitrate_kbps_est": Path(path).stat().st_size * 8 / max(duration, 1e-6) / 1000,
        "rms": float(np.sqrt(np.mean(analysis**2) + 1e-12)),
        "clipping_ratio": float(np.mean(np.abs(analysis) >= 0.999)),
        "zcr": float(librosa.feature.zero_crossing_rate(analysis)[0].mean()),
        "silence_ratio": silence_ratio, "snr_proxy_db": float(snr_proxy),
        "high_frequency_ratio": high_ratio,
        "group_delay_std": float(np.nanstd(group_delay)),
        "instantaneous_frequency_std": float(np.nanstd(instantaneous_frequency)),
        "spectral_flatness": flatness, "harmonic_ratio": harmonic_ratio,
        "onset_rate": onset_rate, "content_hint": content_hint,
    }


from tqdm.auto import tqdm  # 이 EDA 셀만 다시 실행해도 progress bar를 사용할 수 있게 유지

eda_rows, eda_errors = [], []
for row in tqdm(eda_sample.itertuples(index=False), total=len(eda_sample), desc="FoR advanced EDA"):
    try:
        eda_rows.append({"id": row.id, **audio_artifacts(row.path)})
    except Exception as exc:
        eda_errors.append((row.path, repr(exc)))

eda = eda_sample.merge(pd.DataFrame(eda_rows), on="id", how="inner")
print("read errors:", len(eda_errors), eda_errors[:3])
display(eda.describe(include="all").T)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(17, 13))
sns.countplot(data=manifest, x="variant", hue="class_name", ax=axes[0, 0])
axes[0, 0].tick_params(axis="x", rotation=20)
sns.histplot(data=eda, x="duration", hue="variant", bins=45, ax=axes[0, 1])
sns.countplot(data=eda, x="sample_rate", hue="variant", ax=axes[0, 2])
sns.countplot(data=eda, x="channels", hue="variant", ax=axes[1, 0])
sns.boxplot(data=eda, x="variant", y="bitrate_kbps_est", hue="class_name", showfliers=False, ax=axes[1, 1])
sns.boxplot(data=eda, x="variant", y="silence_ratio", hue="class_name", showfliers=False, ax=axes[1, 2])
sns.boxplot(data=eda, x="variant", y="high_frequency_ratio", hue="class_name", showfliers=False, ax=axes[2, 0])
sns.boxplot(data=eda, x="variant", y="group_delay_std", hue="class_name", showfliers=False, ax=axes[2, 1])
sns.boxplot(data=eda, x="variant", y="instantaneous_frequency_std", hue="class_name", showfliers=False, ax=axes[2, 2])
for axis in axes.flat:
    axis.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
print("[codec / container]")
display(pd.crosstab([eda["variant"], eda["codec_hint"]], eda["class_name"], margins=True))

print("[content heuristic — 정답 레이블이 아님]")
display(pd.crosstab([eda["variant"], eda["content_hint"]], eda["class_name"], normalize="index").round(3))

print("[attack metadata coverage]")
display(pd.crosstab(eda["attack_hint"], eda["class_name"], margins=True))

summary = eda.groupby(["variant", "class_name"]).agg(
    n=("id", "size"), duration=("duration", "median"), bitrate=("bitrate_kbps_est", "median"),
    silence=("silence_ratio", "mean"), high_freq=("high_frequency_ratio", "mean"),
    phase_gd=("group_delay_std", "mean"), phase_if=("instantaneous_frequency_std", "mean"),
    snr_proxy=("snr_proxy_db", "median"),
).round(4)
display(summary)

if set(eda["attack_hint"]) <= {"real", "synthetic_unknown"}:
    print("주의: 파일 경로에 generator 정보가 없어 TTS/VC/inpainting별 성능을 분리할 수 없습니다.")
if not eda["codec_hint"].isin(["g711", "amr"]).any():
    print("주의: 원본에 G.711/AMR 파일이 확인되지 않았습니다. 아래 codec probe와 RawBoost로 별도 모사합니다.")


## 5. 스펙트럼·multi-scale 시각화

같은 표시 범위를 사용해 real/fake 및 각 variant를 비교합니다. `for-rerec`에서만 나타나는 대역 제한을 fake 특징으로 오인하지 않는지 확인하세요.


In [ ]:
picks = []
for _, group in eda.groupby(["variant", "class_name"], sort=False):
    picks.append(group.sample(1, random_state=CFG.seed))
picks = pd.concat(picks, ignore_index=True).head(8)

fig, axes = plt.subplots(len(picks), 2, figsize=(14, 2.6 * len(picks)), squeeze=False)
for index, row in picks.iterrows():
    wav, _ = librosa.load(row["path"], sr=CFG.sample_rate, mono=True, duration=6)
    axes[index, 0].plot(np.arange(len(wav)) / CFG.sample_rate, wav, linewidth=0.7)
    axes[index, 0].set_title(f'{row["variant"]} / {row["class_name"]}')
    mel = librosa.feature.melspectrogram(y=wav, sr=CFG.sample_rate, n_fft=1024, hop_length=160, n_mels=96)
    librosa.display.specshow(librosa.power_to_db(mel, ref=np.max), sr=CFG.sample_rate, hop_length=160, x_axis="time", y_axis="mel", ax=axes[index, 1])
plt.tight_layout()
plt.show()

example_path = picks.iloc[0]["path"]
wav, _ = librosa.load(example_path, sr=CFG.sample_rate, mono=True, duration=8)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for axis, n_fft, hop in zip(axes, (256, 1024, 2048), (80, 256, 512)):
    spec = np.abs(librosa.stft(wav, n_fft=n_fft, hop_length=hop))**2
    librosa.display.specshow(librosa.power_to_db(spec, ref=np.max), sr=CFG.sample_rate, hop_length=hop, x_axis="time", y_axis="linear", ax=axis)
    axis.set_title(f"n_fft={n_fft}, hop={hop}")
plt.tight_layout()
plt.show()


## 6. 선택적 실제 codec probe

FFmpeg encoder/decoder를 실제 통과시켜 MP3, OPUS, G.711 μ-law/A-law, AMR-NB가 EDA 특징을 어떻게 바꾸는지 확인합니다. Colab FFmpeg에 AMR encoder가 없으면 해당 항목만 건너뜁니다.


In [ ]:
RUN_CODEC_PROBE = False

if RUN_CODEC_PROBE:
    codec_settings = {
        "mp3_64k": (".mp3", ["-codec:a", "libmp3lame", "-b:a", "64k"]),
        "opus_32k": (".ogg", ["-codec:a", "libopus", "-b:a", "32k"]),
        "g711_mulaw": (".wav", ["-codec:a", "pcm_mulaw", "-ar", "8000"]),
        "g711_alaw": (".wav", ["-codec:a", "pcm_alaw", "-ar", "8000"]),
        "amr_nb": (".amr", ["-codec:a", "libopencore_amrnb", "-ar", "8000", "-ac", "1", "-b:a", "12.2k"]),
    }
    probe_source = Path(picks.iloc[0]["path"])
    probe_rows = [{"codec": "original", **audio_artifacts(str(probe_source))}]
    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        for codec_name, (suffix, args) in codec_settings.items():
            encoded = temporary / f"encoded{suffix}"
            decoded = temporary / f"{codec_name}.wav"
            try:
                subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(probe_source), "-ac", "1", *args, str(encoded)], check=True, timeout=90)
                subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(encoded), "-ac", "1", "-ar", str(CFG.sample_rate), str(decoded)], check=True, timeout=90)
                probe_rows.append({"codec": codec_name, **audio_artifacts(str(decoded))})
            except Exception as exc:
                print("skip", codec_name, repr(exc))
    display(pd.DataFrame(probe_rows)[[
        "codec", "bitrate_kbps_est", "silence_ratio", "high_frequency_ratio",
        "group_delay_std", "instantaneous_frequency_std", "snr_proxy_db",
    ]].round(4))


## 7. 학습/selection/audit 구성

- Train: 네 variant의 전체 `training`
- Selection: 네 variant의 전체 `validation`
- Untouched audit: 네 variant의 전체 `testing`

190,000여 개를 한 덩어리로 섞어 train/test 누수를 만드는 대신, 원 제작자가 제공한 공식 split을 지킵니다. 즉 “전체 파일 학습”은 모든 training 파일을 뜻하며 validation/testing 파일은 학습하지 않습니다. 네 variant별 성능을 따로 출력해 원본·정규화·2초·재녹음 환경에서 성능이 유지되는지 확인합니다.


In [ ]:
ALL_VARIANTS = ["for-original", "for-norm", "for-2sec", "for-rerec"]
TRAIN_VARIANTS = ALL_VARIANTS
SELECTION_VARIANTS = ALL_VARIANTS
AUDIT_VARIANTS = ALL_VARIANTS

train_frame = manifest[(manifest["split"] == "train") & manifest["variant"].isin(TRAIN_VARIANTS)].copy()
selection_frame = manifest[(manifest["split"] == "validation") & manifest["variant"].isin(SELECTION_VARIANTS)].copy()
audit_frame = manifest[(manifest["split"] == "test") & manifest["variant"].isin(AUDIT_VARIANTS)].copy()

expected_train = int((manifest["split"] == "train").sum())
expected_validation = int((manifest["split"] == "validation").sum())
expected_test = int((manifest["split"] == "test").sum())
assert len(train_frame) == expected_train, (len(train_frame), expected_train)
assert len(selection_frame) == expected_validation, (len(selection_frame), expected_validation)
assert len(audit_frame) == expected_test, (len(audit_frame), expected_test)
assert set(train_frame["label"]) == {0, 1}
assert set(selection_frame["label"]) == {0, 1}
assert set(audit_frame["label"]) == {0, 1}

if set(train_frame["group_id"]) & set(selection_frame["group_id"]):
    raise RuntimeError("train/selection group leakage")
if set(train_frame["group_id"]) & set(audit_frame["group_id"]):
    raise RuntimeError("train/audit group leakage")
if selection_frame["variant"].nunique() < 2:
    raise RuntimeError("교차 domain selection을 위해 validation variant가 2개 이상 필요합니다.")

display(pd.crosstab([train_frame["variant"], train_frame["split"]], train_frame["class_name"], margins=True))
display(pd.crosstab([selection_frame["variant"], selection_frame["split"]], selection_frame["class_name"], margins=True))
display(pd.crosstab([audit_frame["variant"], audit_frame["split"]], audit_frame["class_name"], margins=True))
print(f"전체 인덱스: {len(manifest):,}")
print(f"학습 사용: {len(train_frame):,} / 전체 training {expected_train:,} (100%)")
print(f"검증 사용: {len(selection_frame):,} / 전체 validation {expected_validation:,} (100%)")
print(f"테스트 사용: {len(audit_frame):,} / 전체 testing {expected_test:,} (100%)")


## 8. 공식 AASIST·RawBoost 소스 준비

공식 저장소를 얕게 clone하고 실제 commit hash를 checkpoint에 기록합니다.


In [ ]:
repositories = {
    "aasist": "https://github.com/clovaai/aasist.git",
    "rawboost": "https://github.com/TakHemlata/RawBoost-antispoofing.git",
}
repo_commits = {}
for name, url in repositories.items():
    destination = REPO_ROOT / name
    if not destination.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)
    repo_commits[name] = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True
    ).strip()
print(repo_commits)


## 9. 공통 Dataset과 train-only augmentation

필수 함수만 남기기 위해 로딩·길이 조절·증강을 하나의 Dataset class에 모았습니다. Validation/Audit에는 어떤 랜덤 증강도 적용하지 않습니다.


In [ ]:
rawboost_path = REPO_ROOT / "rawboost" / "RawBoost.py"
rawboost_spec = importlib.util.spec_from_file_location("official_rawboost", rawboost_path)
RAWBOOST = importlib.util.module_from_spec(rawboost_spec)
assert rawboost_spec.loader is not None
rawboost_spec.loader.exec_module(RAWBOOST)

AUDIO_ERROR_LOG = []


class FoRAudioDataset(Dataset):
    def __init__(self, frame, clip_samples, train=False, rawboost_p=0.0, communication_p=0.0):
        self.frame = frame.reset_index(drop=True)
        self.clip_samples = clip_samples
        self.train = train
        self.rawboost_p = rawboost_p
        self.communication_p = communication_p

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        soundfile_error = None
        try:
            # 긴 원본 파일 전체를 RAM에 올리지 않고 필요한 구간만 읽습니다.
            with sf.SoundFile(row.path) as audio_file:
                sr = audio_file.samplerate
                required = int(math.ceil(self.clip_samples * sr / CFG.sample_rate)) + 16
                if audio_file.frames > required:
                    maximum_start = audio_file.frames - required
                    source_start = random.randint(0, maximum_start) if self.train else maximum_start // 2
                    audio_file.seek(source_start)
                    wav = audio_file.read(required, dtype="float32", always_2d=True)
                else:
                    wav = audio_file.read(dtype="float32", always_2d=True)
            wav = torch.from_numpy(wav.mean(axis=1))
        except Exception as exc:
            soundfile_error = repr(exc)
            try:
                wav, sr = librosa.load(row.path, sr=None, mono=True)
                wav = torch.from_numpy(np.asarray(wav, dtype=np.float32))
            except Exception as librosa_exc:
                try:
                    # 일부 MP3는 libsndfile/audioread 대신 FFmpeg pipe로 복구됩니다.
                    duration = self.clip_samples / CFG.sample_rate + 1.0
                    decoded = subprocess.run(
                        [
                            "ffmpeg", "-hide_banner", "-loglevel", "error",
                            "-i", row.path, "-t", str(duration), "-ac", "1",
                            "-ar", str(CFG.sample_rate), "-f", "f32le", "pipe:1",
                        ],
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                        check=True, timeout=30,
                    )
                    wav = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())
                    sr = CFG.sample_rate
                    if wav.numel() == 0:
                        raise ValueError("FFmpeg returned empty audio")
                except Exception as ffmpeg_exc:
                    message = (
                        f"soundfile={soundfile_error}; librosa={repr(librosa_exc)}; "
                        f"ffmpeg={repr(ffmpeg_exc)}"
                    )
                    return {
                        "audio": torch.zeros(self.clip_samples, dtype=torch.float32),
                        "label": torch.tensor(int(row.label)), "id": row.id, "domain": row.variant,
                        "path": row.path, "valid": torch.tensor(False), "error": message,
                    }

        try:
            if wav.numel() == 0:
                raise ValueError("decoded audio is empty")
            if sr != CFG.sample_rate:
                wav = torchaudio.functional.resample(wav, sr, CFG.sample_rate)

            if self.train and random.random() < self.rawboost_p:
                values = wav.numpy()
                # 공식 RawBoost algorithm 5: LnL(1) 다음 ISD(2)를 순차 적용
                values = RAWBOOST.LnL_convolutive_noise(
                    values, N_f=5, nBands=5, minF=20, maxF=8000,
                    minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
                    minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20, fs=CFG.sample_rate,
                )
                values = RAWBOOST.ISD_additive_noise(values, P=10, g_sd=2)
                wav = torch.from_numpy(np.asarray(RAWBOOST.normWav(values, 0), dtype=np.float32))

            if self.train and random.random() < self.communication_p:
                mode = random.choice(["telephone", "mulaw", "noise", "gain_clip"])
                if mode == "telephone":
                    wav = torchaudio.functional.resample(wav, CFG.sample_rate, 8000)
                    wav = torchaudio.functional.resample(wav, 8000, CFG.sample_rate)
                elif mode == "mulaw":
                    encoded = torchaudio.functional.mu_law_encoding(wav.clamp(-1, 1), 256)
                    wav = torchaudio.functional.mu_law_decoding(encoded, 256)
                elif mode == "noise":
                    power = wav.square().mean().clamp_min(1e-8)
                    snr = random.uniform(12, 35)
                    wav = wav + torch.randn_like(wav) * (power / 10 ** (snr / 10)).sqrt()
                else:
                    wav = wav * 10 ** (random.uniform(-8, 5) / 20)
                    limit = random.uniform(0.35, 0.95)
                    wav = wav.clamp(-limit, limit) / limit

            if len(wav) >= self.clip_samples:
                start = random.randint(0, len(wav) - self.clip_samples) if self.train else (len(wav) - self.clip_samples) // 2
                wav = wav[start:start + self.clip_samples]
            else:
                wav = wav.repeat(math.ceil(self.clip_samples / len(wav)))[:self.clip_samples]
            if not torch.isfinite(wav).all():
                raise ValueError("decoded audio contains NaN or Inf")
        except Exception as processing_exc:
            return {
                "audio": torch.zeros(self.clip_samples, dtype=torch.float32),
                "label": torch.tensor(int(row.label)), "id": row.id, "domain": row.variant,
                "path": row.path, "valid": torch.tensor(False),
                "error": f"processing={repr(processing_exc)}; soundfile={soundfile_error}",
            }

        return {
            "audio": wav.float().clamp(-1, 1),
            "label": torch.tensor(int(row.label)), "id": row.id, "domain": row.variant,
            "path": row.path, "valid": torch.tensor(True), "error": "",
        }


## 10. 공통 지표

EER score 방향은 `P(real)`입니다. Accuracy만 보면 클래스 불균형을 놓칠 수 있어 balanced accuracy, F1, AUC도 항상 저장합니다.


In [ ]:
def binary_report(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores)
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    index = int(np.argmin(np.abs(fnr - fpr)))
    eer = float((fpr[index] + fnr[index]) / 2)
    predicted = (scores >= 0.5).astype(int)
    return {
        "eer": eer, "eer_threshold": float(thresholds[index]),
        "accuracy": accuracy_score(labels, predicted),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "f1": f1_score(labels, predicted, zero_division=0),
        "auc": roc_auc_score(labels, scores),
    }


## 11. [1–2] Tiny Log-Mel baseline

가장 먼저 실행할 sanity model입니다. 이 모델이 한 epoch도 학습되지 않으면 고급 모델로 넘어가지 않습니다.


In [ ]:
class TinyLogMel(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, win_length=400,
            hop_length=160, n_mels=96, f_min=20, f_max=7600,
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 2))

    def forward(self, audio):
        feature = torch.log(self.mel(audio).clamp_min(1e-6))
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)
        return self.head(self.encoder(feature.unsqueeze(1)))


## 12. [3–4] 공식 AASIST와 AASIST+RawBoost

AASIST 구조는 공식 저장소 구현을 불러옵니다. RawBoost와 통신 증강은 Dataset에서 train에만 적용합니다.


In [ ]:
def official_aasist(freq_aug=False):
    repo = REPO_ROOT / "aasist"
    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)

        def forward(self, audio):
            _, logits = self.net(audio, Freq_aug=freq_aug and self.training)
            return logits

    return Wrapper()


## 13. [5–7] XLS-R, AASIST-inspired dual graph, TCM

- `XLSRPool`: multilingual SSL representation baseline
- `XLSRDualGraph`: XLS-R frame에 시간/특징 graph view를 적용한 AASIST-inspired 모델
- `XLSRTCM`: temporal depthwise convolution + channel gate + self-attention

`XLSRDualGraph`는 공식 raw-waveform AASIST와 동일한 모델이 아니라 결합 아이디어를 재구현한 것입니다.


In [ ]:
from transformers import AutoModel


class SSLBase(nn.Module):
    def __init__(self, model_name, freeze=True):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name)
        self.hidden = self.ssl.config.hidden_size
        if freeze:
            for parameter in self.ssl.parameters():
                parameter.requires_grad = False

    def features(self, audio):
        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)
        if any(parameter.requires_grad for parameter in self.ssl.parameters()):
            return self.ssl(audio).last_hidden_state
        with torch.no_grad():
            return self.ssl(audio).last_hidden_state

    def unfreeze_last(self, count=4):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False
        layers = self.ssl.encoder.layers
        for layer in layers[-count:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True


class AttentivePool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x):
        weights = torch.softmax(self.attention(x), dim=1)
        mean = (x * weights).sum(1)
        std = (weights * (x - mean[:, None]).square()).sum(1).clamp_min(1e-6).sqrt()
        return torch.cat([mean, std], dim=-1)


class XLSRPool(SSLBase):
    def __init__(self, model_name, freeze=True, dropout=0.2):
        super().__init__(model_name, freeze)
        self.pool = AttentivePool(self.hidden)
        self.head = nn.Sequential(nn.LayerNorm(self.hidden * 2), nn.Dropout(dropout), nn.Linear(self.hidden * 2, 256), nn.GELU(), nn.Linear(256, 2))

    def forward(self, audio):
        return self.head(self.pool(self.features(audio)))


class AttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, 4, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attention(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRDualGraph(SSLBase):
    def __init__(self, model_name, freeze=True, dim=128, dropout=0.2):
        super().__init__(model_name, freeze)
        self.projection = nn.Linear(self.hidden, dim)
        self.feature_projection = nn.Linear(8, dim)
        self.time_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.feature_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.head = nn.Sequential(nn.LayerNorm(dim * 4), nn.Dropout(dropout), nn.Linear(dim * 4, 256), nn.GELU(), nn.Linear(256, 2))

    def forward(self, audio):
        hidden = self.projection(self.features(audio))
        time_nodes = F.adaptive_avg_pool1d(hidden.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = self.feature_projection(F.adaptive_avg_pool1d(hidden.transpose(1, 2), 8))
        time_nodes = self.time_graph(time_nodes)
        feature_nodes = self.feature_graph(feature_nodes)
        pooled = torch.cat([time_nodes.mean(1), time_nodes.amax(1), feature_nodes.mean(1), feature_nodes.amax(1)], dim=-1)
        return self.head(pooled)


class TCMBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.temporal = nn.Conv1d(dim, dim, 7, padding=3, groups=dim)
        self.gate = nn.Sequential(nn.Linear(dim, dim // 8), nn.SiLU(), nn.Linear(dim // 8, dim), nn.Sigmoid())
        self.attention = AttentionBlock(dim, dropout)

    def forward(self, x):
        z = self.norm(x)
        x = x + self.temporal(z.transpose(1, 2)).transpose(1, 2) * self.gate(z.mean(1)).unsqueeze(1)
        return self.attention(x)


class XLSRTCM(SSLBase):
    def __init__(self, model_name, freeze=True, dim=256, dropout=0.2):
        super().__init__(model_name, freeze)
        self.projection = nn.Linear(self.hidden, dim)
        self.tcm = nn.Sequential(TCMBlock(dim), TCMBlock(dim), TCMBlock(dim))
        self.pool = AttentivePool(dim)
        self.head = nn.Sequential(nn.LayerNorm(dim * 2), nn.Dropout(dropout), nn.Linear(dim * 2, 2))

    def forward(self, audio):
        return self.head(self.pool(self.tcm(self.projection(self.features(audio)))))


## 14. [8] Speech/Spectral multi-branch

FoR에는 music presence 정답이 없으므로 이 모델을 음성/음악 multi-task classifier라고 부르지 않습니다. XLS-R의 speech-context branch와 multi-scale spectrogram branch를 결합해 음악·배경음이 섞인 외부 환경에도 쓸 수 있는 표현 다양성을 확보합니다.


In [ ]:
class XLSRSpectralFusion(SSLBase):
    def __init__(self, model_name, freeze=True, dropout=0.25):
        super().__init__(model_name, freeze)
        self.ssl_pool = AttentivePool(self.hidden)
        self.mels = nn.ModuleList([
            torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sample_rate, n_fft=n_fft, hop_length=hop, n_mels=96, f_max=7600)
            for n_fft, hop in ((512, 160), (1024, 256), (2048, 512))
        ])
        self.spectral = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.GELU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.LayerNorm(self.hidden * 2 + 128), nn.Dropout(dropout), nn.Linear(self.hidden * 2 + 128, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 2))

    def forward(self, audio):
        ssl_embedding = self.ssl_pool(self.features(audio))
        mel_features = []
        for transform in self.mels:
            feature = torch.log(transform(audio).clamp_min(1e-6))
            feature = F.interpolate(feature.unsqueeze(1), size=(96, 256), mode="bilinear", align_corners=False)
            mel_features.append(feature)
        spectral_embedding = self.spectral(torch.cat(mel_features, dim=1))
        return self.head(torch.cat([ssl_embedding, spectral_embedding], dim=-1))


## 15. 모델별 권장 시작 설정

| 단계 | 모델 | clip | batch/accum | epoch | head LR | SSL LR | 증강 |
|---|---|---:|---:|---:|---:|---:|---|
| 2 | Tiny Log-Mel | 4.0s | 32/1 | 18 | 3e-4 | - | none |
| 3 | AASIST | 4.04s | 16/1 | 25 | 1e-4 | - | none |
| 4 | AASIST+RawBoost | 4.04s | 16/1 | 30 | 1e-4 | - | RawBoost .45 + 통신 .25 |
| 5 | XLS-R pool | 4.0s | 2/8 | 12 | 1e-4 | 5e-7 | freeze 2 epoch |
| 6 | XLS-R dual graph | 4.0s | 2/8 | 14 | 8e-5 | 3e-7 | freeze 3 epoch |
| 7 | XLS-R TCM | 4.0s | 2/8 | 14 | 8e-5 | 3e-7 | freeze 3 epoch |
| 8 | XLS-R spectral fusion | 4.0s | 2/8 | 14 | 8e-5 | 3e-7 | 통신 .2 |

T4에서 메모리가 부족하면 batch만 절반으로 줄이고 `grad_accum`을 두 배로 올려 effective batch를 유지합니다.


In [ ]:
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"

EXPERIMENTS = {
    "exp01_tiny": dict(model="tiny", clip_samples=64_000, batch=32, eval_batch=64, grad_accum=1, epochs=18, lr=3e-4, backbone_lr=3e-4, weight_decay=1e-4, patience=5, dropout=0.25),
    "exp02_aasist": dict(model="aasist", clip_samples=64_600, batch=16, eval_batch=32, grad_accum=1, epochs=25, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4, patience=6, dropout=0.2),
    "exp03_aasist_rawboost": dict(model="aasist", clip_samples=64_600, batch=16, eval_batch=32, grad_accum=1, epochs=30, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4, patience=7, rawboost_p=0.45, communication_p=0.25, dropout=0.2),
    "exp04_xlsr_pool": dict(model="xlsr_pool", clip_samples=64_000, batch=2, eval_batch=4, grad_accum=8, epochs=12, lr=1e-4, backbone_lr=5e-7, weight_decay=1e-4, patience=5, freeze_epochs=2, unfreeze_last=4, dropout=0.2),
    "exp05_xlsr_dualgraph": dict(model="xlsr_dualgraph", clip_samples=64_000, batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, dropout=0.2),
    "exp06_xlsr_tcm": dict(model="xlsr_tcm", clip_samples=64_000, batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, rawboost_p=0.20, communication_p=0.25, dropout=0.2),
    "exp07_multibranch": dict(model="multibranch", clip_samples=64_000, batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, communication_p=0.20, dropout=0.25),
}
display(pd.DataFrame(EXPERIMENTS).T)


## 16. 공통 model/loader/training

각 epoch는 training 파일 전체를 정확히 한 번 순회합니다. 클래스 불균형은 중복 sampling이 아니라 weighted cross-entropy로 처리합니다. `best.pt`, 재시작용 `last.pt`, history, selection prediction은 Drive에 저장됩니다.


In [ ]:
RESUME_TRAINING = True


def build_model(exp):
    if exp["model"] == "tiny":
        return TinyLogMel(exp.get("dropout", 0.25))
    if exp["model"] == "aasist":
        return official_aasist(False)
    if exp["model"] == "xlsr_pool":
        return XLSRPool(XLSR_MODEL, True, exp["dropout"])
    if exp["model"] == "xlsr_dualgraph":
        return XLSRDualGraph(XLSR_MODEL, True, dropout=exp["dropout"])
    if exp["model"] == "xlsr_tcm":
        return XLSRTCM(XLSR_MODEL, True, dropout=exp["dropout"])
    if exp["model"] == "multibranch":
        return XLSRSpectralFusion(XLSR_MODEL, True, exp["dropout"])
    raise KeyError(exp["model"])


def make_loaders(exp):
    train_dataset = FoRAudioDataset(
        train_frame, exp["clip_samples"], train=True,
        rawboost_p=exp.get("rawboost_p", 0.0), communication_p=exp.get("communication_p", 0.0),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=exp["batch"],
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=CFG.num_workers > 0,
        drop_last=False,
    )
    validation_loaders = {}
    for domain, frame in selection_frame.groupby("variant"):
        dataset = FoRAudioDataset(frame, exp["clip_samples"], train=False)
        validation_loaders[domain] = DataLoader(
            dataset,
            batch_size=exp["eval_batch"],
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=DEVICE.type == "cuda",
            persistent_workers=CFG.num_workers > 0,
        )
    return train_loader, validation_loaders


def run_loader(
    model, loader, criterion, optimizer=None, scheduler=None, scaler=None,
    grad_accum=1, description=None,
):
    training = optimizer is not None
    model.train(training)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss, sample_count, skipped = 0.0, 0, 0
    labels, scores, identifiers = [], [], []
    error_examples = []
    progress = tqdm(
        loader,
        total=len(loader),
        leave=True,
        dynamic_ncols=True,
        desc=description or ("train" if training else "valid"),
    )
    for step, batch in enumerate(progress, 1):
        valid_mask = batch["valid"].bool()
        valid_flags = valid_mask.tolist()
        invalid_in_batch = len(valid_flags) - sum(valid_flags)
        if invalid_in_batch:
            skipped += invalid_in_batch
            for keep, identifier, path, error in zip(
                valid_flags, batch["id"], batch["path"], batch["error"]
            ):
                if not keep:
                    item = {
                        "stage": description or ("train" if training else "valid"),
                        "id": identifier, "path": path, "error": error,
                    }
                    AUDIO_ERROR_LOG.append(item)
                    if len(error_examples) < 5:
                        error_examples.append(item)
        if not valid_mask.any():
            progress.set_postfix(samples=f"{sample_count:,}", skipped=skipped)
            continue

        audio = batch["audio"][valid_mask].to(DEVICE, non_blocking=True)
        target = batch["label"][valid_mask].to(DEVICE, non_blocking=True)
        valid_identifiers = [identifier for identifier, keep in zip(batch["id"], valid_flags) if keep]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(audio)
            unscaled_loss = criterion(logits, target)
            loss = unscaled_loss / (grad_accum if training else 1)
        if training:
            scaler.scale(loss).backward()
            if step % grad_accum == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
        batch_size = int(target.shape[0])
        total_loss += float(unscaled_loss.detach().cpu()) * batch_size
        sample_count += batch_size
        labels.extend(target.detach().cpu().numpy().tolist())
        scores.extend(torch.softmax(logits.detach(), dim=-1)[:, 1].cpu().numpy().tolist())
        identifiers.extend(valid_identifiers)
        if step == 1 or step % 50 == 0 or step == len(loader):
            progress.set_postfix(
                loss=f"{total_loss / max(1, sample_count):.4f}",
                samples=f"{sample_count:,}", skipped=skipped,
            )
    if not labels:
        raise RuntimeError(f"{description}: 읽을 수 있는 오디오가 하나도 없습니다.")
    report = binary_report(labels, scores)
    report["loss"] = total_loss / max(1, sample_count)
    report["samples"] = sample_count
    report["skipped"] = skipped
    if skipped:
        error_log_path = DRIVE_ROOT / "audio_read_errors.csv"
        pd.DataFrame(AUDIO_ERROR_LOG).drop_duplicates(["stage", "id", "error"]).to_csv(error_log_path, index=False)
        print(f"경고: {description}에서 읽지 못한 오디오 {skipped:,}개를 제외했습니다.")
        display(pd.DataFrame(error_examples))
        print("saved:", error_log_path)
    return report, np.asarray(labels), np.asarray(scores), identifiers


def fit_experiment(experiment_name):
    from transformers import get_cosine_schedule_with_warmup

    exp = dict(EXPERIMENTS[experiment_name])
    run_dir = RUN_ROOT / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_loader, validation_loaders = make_loaders(exp)
    model = build_model(exp).to(DEVICE)

    backbone, head = [], []
    for name, parameter in model.named_parameters():
        (backbone if name.startswith("ssl.") else head).append(parameter)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": exp["backbone_lr"]})
    if head:
        groups.append({"params": head, "lr": exp["lr"]})
    optimizer = torch.optim.AdamW(groups, weight_decay=exp["weight_decay"])
    updates_per_epoch = math.ceil(len(train_loader) / exp["grad_accum"])
    total_updates = max(1, updates_per_epoch * exp["epochs"])
    scheduler = get_cosine_schedule_with_warmup(optimizer, max(10, int(total_updates * 0.08)), total_updates)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    class_counts = train_frame["label"].value_counts().reindex([0, 1], fill_value=0).to_numpy()
    class_weights = len(train_frame) / (2 * np.maximum(class_counts, 1))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))

    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"[{experiment_name}] 전체 학습 파일={len(train_frame):,}, "
        f"batch/epoch={len(train_loader):,}, epochs={exp['epochs']}, "
        f"parameters={total_parameters:,}, initially_trainable={trainable:,}"
    )
    print("class counts:", dict(enumerate(class_counts.tolist())), "loss weights:", class_weights.round(4).tolist())

    best_score, stale, history, start_epoch = float("inf"), 0, [], 1
    last_path = run_dir / "last.pt"
    if RESUME_TRAINING and last_path.exists():
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        if state.get("config") == exp:
            model.load_state_dict(state["model_state"], strict=True)
            if state["epoch"] >= exp.get("freeze_epochs", 10**9) and hasattr(model, "unfreeze_last"):
                model.unfreeze_last(exp.get("unfreeze_last", 4))
            optimizer.load_state_dict(state["optimizer_state"])
            scheduler.load_state_dict(state["scheduler_state"])
            scaler.load_state_dict(state["scaler_state"])
            best_score = float(state["best_score"])
            stale = int(state["stale"])
            history = list(state["history"])
            start_epoch = int(state["epoch"]) + 1
            print(f"last.pt에서 재개: 완료 epoch={state['epoch']}, 다음 epoch={start_epoch}")
            if state.get("finished", False):
                print("이미 full training이 완료된 checkpoint입니다.")
                del model
                gc.collect()
                torch.cuda.empty_cache()
                return pd.DataFrame(history)
        else:
            print("기존 last.pt 설정이 현재 설정과 달라 새로 학습합니다.")

    for epoch in range(start_epoch, exp["epochs"] + 1):
        epoch_started = time.time()
        print(f"\n[{experiment_name}] epoch {epoch}/{exp['epochs']} 시작")
        if epoch == exp.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last"):
            model.unfreeze_last(exp.get("unfreeze_last", 4))
            print("unfroze SSL layers")
        train_metrics, *_ = run_loader(
            model, train_loader, criterion, optimizer, scheduler, scaler,
            exp["grad_accum"], f"{experiment_name} train {epoch}/{exp['epochs']}",
        )

        domain_rows, prediction_rows = [], []
        with torch.no_grad():
            for domain, loader in validation_loaders.items():
                metrics, labels, scores, identifiers = run_loader(
                    model, loader, criterion,
                    description=f"{experiment_name} validation {domain}",
                )
                domain_rows.append({"domain": domain, **metrics})
                prediction_rows.append(pd.DataFrame({"id": identifiers, "domain": domain, "label": labels, "score_real": scores}))
        domain_report = pd.DataFrame(domain_rows)
        robust_score = domain_report["eer"].mean() + 0.5 * domain_report["eer"].max() + 0.1 * domain_report["eer"].std(ddof=0)
        record = {
            "epoch": epoch, "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"], "train_eer": train_metrics["eer"],
            "train_samples": train_metrics["samples"], "train_skipped": train_metrics["skipped"],
            "mean_eer": domain_report["eer"].mean(), "worst_eer": domain_report["eer"].max(),
            "mean_accuracy": domain_report["accuracy"].mean(),
            "mean_balanced_accuracy": domain_report["balanced_accuracy"].mean(),
            "mean_f1": domain_report["f1"].mean(), "mean_auc": domain_report["auc"].mean(),
            "robust_score": robust_score, "minutes": (time.time() - epoch_started) / 60,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print("validation metrics by domain:")
        display(domain_report)
        print("epoch summary:", {key: round(value, 6) if isinstance(value, float) else value for key, value in record.items()})

        if robust_score < best_score:
            best_score, stale = robust_score, 0
            torch.save({
                "model_state": model.state_dict(), "experiment": experiment_name, "config": exp,
                "epoch": epoch, "selection_report": domain_rows, "robust_score": robust_score,
                "training_files": len(train_frame), "validation_files": len(selection_frame),
                "label_convention": {"0": "fake", "1": "real"},
                "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
                "repo_commits": repo_commits,
            }, run_dir / "best.pt")
            pd.concat(prediction_rows, ignore_index=True).to_csv(run_dir / "selection_predictions.csv", index=False)
        else:
            stale += 1

        finished = stale >= exp["patience"] or epoch == exp["epochs"]
        torch.save({
            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
            "experiment": experiment_name, "config": exp, "epoch": epoch,
            "best_score": best_score, "stale": stale, "history": history,
            "finished": finished,
        }, last_path)
        print("saved:", run_dir / "best.pt", "| resume:", last_path)

        if stale >= exp["patience"]:
            print("early stopping")
            break

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(history)


## 17. 모델별 소규모 parameter search

아래 후보는 모든 조합을 무작정 탐색하지 않고 영향이 큰 LR·dropout·증강 확률만 비교합니다. `SEARCH_BASE` 한 모델씩 3–5 epoch로 탐색한 뒤 `robust_score`가 가장 낮은 설정을 원래 experiment에 반영해 full training 하세요.


In [ ]:
PARAMETER_SEARCH = {
    "exp01_tiny": [
        {"lr": 1e-4, "dropout": 0.20}, {"lr": 3e-4, "dropout": 0.25}, {"lr": 7e-4, "dropout": 0.35},
    ],
    "exp02_aasist": [
        {"lr": 5e-5, "weight_decay": 1e-4}, {"lr": 1e-4, "weight_decay": 1e-4}, {"lr": 2e-4, "weight_decay": 5e-4},
    ],
    "exp03_aasist_rawboost": [
        {"rawboost_p": 0.25, "communication_p": 0.15}, {"rawboost_p": 0.45, "communication_p": 0.25}, {"rawboost_p": 0.60, "communication_p": 0.30},
    ],
    "exp04_xlsr_pool": [
        {"lr": 5e-5, "backbone_lr": 3e-7}, {"lr": 1e-4, "backbone_lr": 5e-7}, {"lr": 2e-4, "backbone_lr": 1e-6},
    ],
    "exp05_xlsr_dualgraph": [
        {"lr": 5e-5, "dropout": 0.15}, {"lr": 8e-5, "dropout": 0.20}, {"lr": 1.5e-4, "dropout": 0.30},
    ],
    "exp06_xlsr_tcm": [
        {"rawboost_p": 0.0, "communication_p": 0.15}, {"rawboost_p": 0.20, "communication_p": 0.25}, {"rawboost_p": 0.35, "communication_p": 0.35},
    ],
    "exp07_multibranch": [
        {"lr": 5e-5, "dropout": 0.20}, {"lr": 8e-5, "dropout": 0.25}, {"lr": 1.2e-4, "dropout": 0.35},
    ],
}

RUN_PARAMETER_SEARCH = False
SEARCH_BASE = "exp01_tiny"

if RUN_PARAMETER_SEARCH:
    search_rows = []
    for trial_index, override in enumerate(PARAMETER_SEARCH[SEARCH_BASE]):
        trial_name = f"{SEARCH_BASE}_search_{trial_index:02d}"
        EXPERIMENTS[trial_name] = {**EXPERIMENTS[SEARCH_BASE], **override, "epochs": 5, "patience": 5}
        trial_history = fit_experiment(trial_name)
        search_rows.append({"trial": trial_name, **override, "best_robust_score": trial_history["robust_score"].min()})
    search_result = pd.DataFrame(search_rows).sort_values("best_robust_score")
    search_result.to_csv(RUN_ROOT / f"{SEARCH_BASE}_parameter_search.csv", index=False)
    display(search_result)


## 18. 단계별 실행

기본값으로 7개 모델을 Tiny → AASIST → RawBoost → SSL 순서로 모두 학습합니다. 표본 제한은 없으며, 각 모델은 네 variant의 training 파일 전체를 매 epoch 사용합니다. Colab 세션이 끊기면 같은 셀을 다시 실행했을 때 각 모델의 `last.pt` 다음 epoch부터 이어집니다.


In [ ]:
RUN_TRAINING = True
EXPERIMENTS_TO_RUN = list(EXPERIMENTS.keys())

if RUN_TRAINING:
    training_summary_rows = []
    for experiment_name in EXPERIMENTS_TO_RUN:
        print("\n===", experiment_name, "===")
        history = fit_experiment(experiment_name)
        display(history)
        best_row = history.loc[history["robust_score"].idxmin()].to_dict()
        training_summary_rows.append({"experiment": experiment_name, **best_row})
    training_summary = pd.DataFrame(training_summary_rows).sort_values("robust_score")
    training_summary.to_csv(RUN_ROOT / "full_training_summary.csv", index=False)
    print("모든 모델의 validation 기준 best epoch 요약")
    display(training_summary)


## 19. Shape smoke test

학습 전 선택한 모델의 입력/출력 크기만 확인합니다. SSL 모델은 최초 실행 시 Hugging Face weight를 내려받습니다.


In [ ]:
RUN_SHAPE_TEST = False
SHAPE_TEST_EXPERIMENT = "exp01_tiny"

if RUN_SHAPE_TEST:
    selected_exp = EXPERIMENTS[SHAPE_TEST_EXPERIMENT]
    loader, _ = make_loaders(selected_exp)
    batch = next(iter(loader))
    test_model = build_model(selected_exp).to(DEVICE).eval()
    with torch.no_grad():
        logits = test_model(batch["audio"][:2].to(DEVICE))
    print("audio:", batch["audio"].shape, "logits:", logits.shape)
    assert logits.shape == (2, 2)
    del test_model
    gc.collect()
    torch.cuda.empty_cache()


## 20. Untouched testing 교차-domain 평가

각 모델의 validation 기준 `best.pt`를 불러와 네 variant의 testing 파일 전체를 자동 평가합니다. Domain별 지표와 모든 testing 파일을 합친 overall 지표를 모두 저장·출력합니다.


In [ ]:
RUN_FINAL_AUDIT = True
AUDIT_EXPERIMENTS = EXPERIMENTS_TO_RUN

if RUN_FINAL_AUDIT:
    all_audit_rows = []
    for experiment_name in AUDIT_EXPERIMENTS:
        checkpoint_path = RUN_ROOT / experiment_name / "best.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"학습 checkpoint가 없습니다: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        exp = checkpoint["config"]
        model = build_model(exp)
        model.load_state_dict(checkpoint["model_state"], strict=True)
        model.to(DEVICE).eval()
        rows, predictions = [], []
        criterion = nn.CrossEntropyLoss()
        for domain, frame in audit_frame.groupby("variant"):
            dataset = FoRAudioDataset(frame, exp["clip_samples"], train=False)
            loader = DataLoader(
                dataset, batch_size=exp["eval_batch"], shuffle=False,
                num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda",
                persistent_workers=CFG.num_workers > 0,
            )
            with torch.no_grad():
                metrics, labels, scores, identifiers = run_loader(
                    model, loader, criterion,
                    description=f"{experiment_name} test {domain}",
                )
            rows.append({"domain": domain, **metrics})
            predictions.append(pd.DataFrame({"id": identifiers, "domain": domain, "label": labels, "score_real": scores}))

        prediction_frame = pd.concat(predictions, ignore_index=True)
        overall_metrics = binary_report(prediction_frame["label"], prediction_frame["score_real"])
        overall_metrics["loss"] = np.nan
        overall_metrics["samples"] = len(prediction_frame)
        overall_metrics["skipped"] = int(sum(row["skipped"] for row in rows))
        rows.append({"domain": "overall", **overall_metrics})
        report = pd.DataFrame(rows)
        report.insert(0, "experiment", experiment_name)
        report.to_csv(RUN_ROOT / experiment_name / "audit_report.csv", index=False)
        prediction_frame.to_csv(RUN_ROOT / experiment_name / "audit_predictions.csv", index=False)
        all_audit_rows.extend(report.to_dict("records"))
        print(f"{experiment_name} 전체 testing 평가 결과")
        display(report)
        del model
        gc.collect()
        torch.cuda.empty_cache()

    audit_summary = pd.DataFrame(all_audit_rows)
    audit_summary.to_csv(RUN_ROOT / "all_models_test_metrics.csv", index=False)
    print("모든 모델 × testing domain 성능표")
    display(audit_summary.sort_values(["domain", "eer"]))

    print("모델별 overall testing 성능 순위")
    display(
        audit_summary[audit_summary["domain"] == "overall"]
        .sort_values(["eer", "accuracy"], ascending=[True, False])
        .reset_index(drop=True)
    )


## 21. [9] Validation ensemble weight 최적화

관점이 다른 모델만 선택합니다. 기본 후보는 `AASIST+RawBoost`, `XLS-R TCM`, `multi-branch`입니다. Validation OOF의 log-loss로 음수가 아니고 합이 1인 weight를 학습하며 TEST 예측으로 weight를 바꾸지 않습니다.


In [ ]:
ENSEMBLE_EXPERIMENTS = ["exp03_aasist_rawboost", "exp06_xlsr_tcm", "exp07_multibranch"]
selection_files = [RUN_ROOT / name / "selection_predictions.csv" for name in ENSEMBLE_EXPERIMENTS]
RUN_ENSEMBLE = True

if RUN_ENSEMBLE:
    if not all(path.exists() for path in selection_files):
        raise FileNotFoundError([str(path) for path in selection_files if not path.exists()])
    frames = []
    for index, path in enumerate(selection_files):
        frame = pd.read_csv(path).rename(columns={"score_real": f"m{index}"})
        frames.append(frame)
    merged = frames[0]
    for index, frame in enumerate(frames[1:], 1):
        merged = merged.merge(frame[["id", "domain", f"m{index}"]], on=["id", "domain"], validate="one_to_one")
    model_columns = [column for column in merged if re.fullmatch(r"m\d+", column)]
    prediction_matrix = merged[model_columns].to_numpy()
    labels = merged["label"].to_numpy()

    result = minimize(
        lambda weights: log_loss(labels, np.clip(prediction_matrix @ weights, 1e-6, 1 - 1e-6), labels=[0, 1]),
        np.full(len(model_columns), 1 / len(model_columns)), method="SLSQP",
        bounds=[(0, 1)] * len(model_columns),
        constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1},
    )
    if not result.success:
        raise RuntimeError(result.message)
    ensemble_weights = result.x
    raw_ensemble = prediction_matrix @ ensemble_weights
    print("weights:", dict(zip(ENSEMBLE_EXPERIMENTS, ensemble_weights.round(4))))
    print("ensemble:", binary_report(labels, raw_ensemble))


## 22. [10] Calibration과 threshold 최적화

Platt calibration과 threshold는 selection 전체에서 하나만 학습합니다. Domain별 threshold는 실제 일반화 성능을 과대평가하므로 사용하지 않습니다. 확률 제출이면 threshold를 적용하지 않고 보정된 확률을 사용합니다.


In [ ]:
if RUN_ENSEMBLE:
    logits = np.log(np.clip(raw_ensemble, 1e-6, 1 - 1e-6) / np.clip(1 - raw_ensemble, 1e-6, 1))
    calibrator = LogisticRegression(C=1.0).fit(logits.reshape(-1, 1), labels)
    calibrated = calibrator.predict_proba(logits.reshape(-1, 1))[:, 1]

    candidates = np.linspace(0.02, 0.98, 481)
    accuracy_values = np.array([accuracy_score(labels, calibrated >= threshold) for threshold in candidates])
    f1_values = np.array([f1_score(labels, calibrated >= threshold, zero_division=0) for threshold in candidates])
    accuracy_threshold = float(candidates[accuracy_values.argmax()])
    f1_threshold = float(candidates[f1_values.argmax()])
    eer_threshold = binary_report(labels, calibrated)["eer_threshold"]

    print("raw:", binary_report(labels, raw_ensemble))
    print("calibrated:", binary_report(labels, calibrated))
    print("thresholds:", {"accuracy": accuracy_threshold, "f1": f1_threshold, "eer": eer_threshold})

    fraction_real, mean_predicted = calibration_curve(labels, calibrated, n_bins=12)
    plt.figure(figsize=(5, 5))
    plt.plot(mean_predicted, fraction_real, marker="o", label="ensemble")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("Predicted P(real)")
    plt.ylabel("Observed real fraction")
    plt.legend()
    plt.show()

    ensemble_metadata = {
        "experiments": ENSEMBLE_EXPERIMENTS,
        "weights": ensemble_weights.tolist(),
        "sample_rate": CFG.sample_rate,
        "thresholds": {"accuracy": accuracy_threshold, "f1": f1_threshold, "eer": eer_threshold},
        "platt_coef": calibrator.coef_.tolist(), "platt_intercept": calibrator.intercept_.tolist(),
        "label_convention": {"0": "fake", "1": "real"},
    }
    (RUN_ROOT / "ensemble.json").write_text(json.dumps(ensemble_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", RUN_ROOT / "ensemble.json")


## 23. Audit ensemble 최종 보고서

각 모델의 audit prediction이 모두 있을 때 한 번 실행합니다. Calibration 계수와 weight는 selection에서 저장한 값만 사용합니다.


In [ ]:
RUN_AUDIT_ENSEMBLE = True

if RUN_AUDIT_ENSEMBLE:
    metadata = json.loads((RUN_ROOT / "ensemble.json").read_text(encoding="utf-8"))
    audit_files = [RUN_ROOT / name / "audit_predictions.csv" for name in metadata["experiments"]]
    audit_frames = []
    for index, path in enumerate(audit_files):
        frame = pd.read_csv(path).rename(columns={"score_real": f"m{index}"})
        audit_frames.append(frame)
    audit_merged = audit_frames[0]
    for index, frame in enumerate(audit_frames[1:], 1):
        audit_merged = audit_merged.merge(frame[["id", "domain", f"m{index}"]], on=["id", "domain"], validate="one_to_one")

    matrix = audit_merged[[f"m{i}" for i in range(len(audit_frames))]].to_numpy()
    raw = matrix @ np.asarray(metadata["weights"])
    raw_logit = np.log(np.clip(raw, 1e-6, 1 - 1e-6) / np.clip(1 - raw, 1e-6, 1))
    calibrated_audit = 1 / (1 + np.exp(-(raw_logit * metadata["platt_coef"][0][0] + metadata["platt_intercept"][0])))
    audit_merged["ensemble_score_real"] = calibrated_audit

    final_rows = []
    for domain, frame in audit_merged.groupby("domain"):
        metrics = binary_report(frame["label"], frame["ensemble_score_real"])
        metrics["accuracy_at_selected_threshold"] = accuracy_score(
            frame["label"], frame["ensemble_score_real"] >= metadata["thresholds"]["accuracy"]
        )
        metrics["f1_at_selected_threshold"] = f1_score(
            frame["label"], frame["ensemble_score_real"] >= metadata["thresholds"]["f1"], zero_division=0
        )
        final_rows.append({"domain": domain, **metrics})
    final_report = pd.DataFrame(final_rows)
    final_report.to_csv(RUN_ROOT / "ensemble_audit_report.csv", index=False)
    audit_merged.to_csv(RUN_ROOT / "ensemble_audit_predictions.csv", index=False)
    display(final_report.sort_values("eer"))


## 24. 권장 최종 선택

1. Tiny CNN은 파이프라인 검증용으로만 사용합니다.
2. AASIST와 AASIST+RawBoost를 비교해 RawBoost가 `for-rerec`를 실제 개선하는지 확인합니다.
3. XLS-R pool → dual graph → TCM 순서로 추가 복잡도의 효과를 검증합니다.
4. Multi-branch가 selection EER을 개선하면서 다른 모델과 예측 상관이 낮을 때만 ensemble에 포함합니다.
5. 최종 앙상블은 보통 `AASIST+RawBoost + XLS-R TCM + spectral fusion` 3개면 충분합니다.
6. 한 모델이 모든 domain에서 우세하면 억지로 ensemble하지 않습니다.

FoR만으로는 음악·VC·inpainting 강건성을 입증할 수 없습니다. 해당 환경을 주장하려면 음악 혼합/부분 조작/VC 데이터셋을 별도 audit domain으로 추가해야 합니다.


## 25. 전체 실행 체크리스트

- [ ] Kaggle 인증 후 20 GB 데이터 다운로드 완료
- [ ] manifest에 unknown variant/split/label 없음
- [ ] split-qualified `group_id`로 train/validation/testing 분리 확인
- [ ] EDA에서 sample rate/channel/bitrate/길이 확인
- [ ] silence/high-frequency/phase가 label 대신 variant만 구분하는지 확인
- [ ] 각 epoch의 progress bar가 전체 training 파일 수에 도달
- [ ] Tiny → AASIST → RawBoost → XLS-R → TCM → multi-branch 전체 학습
- [ ] selection domain robust score로 checkpoint 선택
- [ ] selection prediction만으로 weight/calibration/threshold 결정
- [ ] best checkpoint로 네 domain의 전체 testing 평가와 overall 성능표 생성
